## Feature Engineering

##### Finalizes the model-ready feature set: resolves remaining missing values, selects features based on Step 3 EDA findings, and one-hot encodes categorical variables. Outputs a fully numeric, model-ready dataset.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/finlora_cleaned.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])

print("Shape:", df.shape)
df.columns.tolist()

Shape: (126000, 29)


['transaction_id',
 'account_id',
 'account_type',
 'kyc_tier',
 'timestamp',
 'day_of_week',
 'hour_of_day',
 'description',
 'merchant_name',
 'merchant_category',
 'channel',
 'amount',
 'currency',
 'amount_to_avg_ratio',
 'avg_transaction_amount_30d',
 'transaction_velocity_1h',
 'transaction_country',
 'home_country',
 'is_cross_border',
 'device_id',
 'is_new_device',
 'account_age_days',
 'status',
 'is_fraud',
 'account_holder_name',
 'account_created_date',
 'personal_spend_baseline_usd',
 'has_device_on_record',
 'amount_usd']

## Note on Column Counts Across Stages

This notebook moves through three different "shapes" of the data, and two of them happen
to both have 29 columns — but they are **not the same 29 columns**. To avoid confusion:

| Stage | Shape | What it is |
|---|---|---|
| `finlora_cleaned.csv` (input to this notebook) | 126,000 × 29 | All cleaned columns: IDs, timestamps, raw `amount`/`currency`, engineered numeric signals, categorical fields as plain text, and `is_fraud` |
| After one-hot encoding (intermediate) | 126,000 × 47 | The 29 cleaned columns minus 4 categorical columns, plus 22 new dummy columns |
| `finlora_model_ready.csv` (output of this notebook) | 126,000 × 30 | Only the selected features: 7 numeric + 22 encoded categorical = 29 model features, plus `is_fraud` = 30 total |

## is_new_device

In [3]:
df['is_new_device'] = df['is_new_device'].fillna(0).astype(int) #Fill with 0 (treat "no device" as "not new") — 
# ...reasoning: has_device_on_record  already separately captures the "missing" case, so is_new_device can just answer 
# ... "is this specifically a known device," with missing-device rows defaulting to "not flagged as new."

In [4]:
print(df['is_new_device'].value_counts())
print(df['is_new_device'].isnull().sum())  # should be 0

is_new_device
0    122105
1      3895
Name: count, dtype: int64
0


## Concluding our final feature set

In [5]:
print(df.columns.tolist()) #view

['transaction_id', 'account_id', 'account_type', 'kyc_tier', 'timestamp', 'day_of_week', 'hour_of_day', 'description', 'merchant_name', 'merchant_category', 'channel', 'amount', 'currency', 'amount_to_avg_ratio', 'avg_transaction_amount_30d', 'transaction_velocity_1h', 'transaction_country', 'home_country', 'is_cross_border', 'device_id', 'is_new_device', 'account_age_days', 'status', 'is_fraud', 'account_holder_name', 'account_created_date', 'personal_spend_baseline_usd', 'has_device_on_record', 'amount_usd']


Based on everything we've established, a sort of every column into include or exclude:

Include (numeric, ready as-is):

amount_to_avg_ratio — strong signal (EDA #1)
transaction_velocity_1h — very strong signal (EDA #2)
is_new_device — moderate signal (EDA #3), just finalized
has_device_on_record — no signal alone, but cheap to include; Random Forest can learn to ignore it if it's genuinely useless, and it costs nothing to leave in
is_cross_border — weak signal (EDA #4), still legitimate to include
account_age_days — never explicitly analyzed in EDA, but a sensible fraud-relevant feature (very new accounts often carry more risk) — worth adding to the model even without a dedicated EDA section, since it's simple and plausible
personal_spend_baseline_usd — from the accounts table, similarly plausible and cheap to include

Include (categorical → will one-hot encode next):

account_type, kyc_tier, merchant_category, channel — per your original brief's feature list

Exclude — and here's why each one, explicitly:

transaction_id, account_id, device_id — identifiers, not predictive features; a model can't learn anything general from a unique ID
timestamp, day_of_week, hour_of_day — raw timestamp isn't usable directly; day/hour could be engineered further (e.g. "is this a weekend/night transaction"), but that's not in your original scope, so we'll leave them out for this baseline unless you want to add that
description, merchant_name — free text, not structured features (and merchant_name's missingness pattern was already explained, not a fraud signal itself)
status — excluded due to leakage, confirmed in Step 2
avg_transaction_amount_30d — this is the raw baseline amount_to_avg_ratio was already computed from; including both would be redundant (same information twice), so we keep the ratio and drop the raw baseline
transaction_country, home_country — raw country codes; is_cross_border already distills the relevant signal from these, so including the raw codes too would be largely redundant unless you specifically want country-level effects (out of your stated scope)
currency — not analyzed, and account-level currency is likely tightly correlated with home_country/kyc_tier rather than adding new information
is_fraud — this is your target, not a feature; it goes into y, never into X
account_holder_name, account_created_date — identifying/raw-date info already distilled into account_age_days

In [6]:
numeric_features = [
    'amount_to_avg_ratio',
    'transaction_velocity_1h',
    'is_new_device',
    'has_device_on_record',
    'is_cross_border',
    'account_age_days',
    'personal_spend_baseline_usd'
]

categorical_features = [
    'account_type',
    'kyc_tier',
    'merchant_category',
    'channel'
]

target = 'is_fraud' #defining our final feature columns

In [7]:
all_expected = numeric_features + categorical_features + [target]
missing_cols = [col for col in all_expected if col not in df.columns]
print("Missing columns (should be empty):", missing_cols) # check: confirm every column actually exists before we proceed 

Missing columns (should be empty): []


pd.get_dummies takes each categorical column and converts it into multiple binary (0/1) columns — one per category value. E.g. channel with 6 values becomes 6 new columns like channel_Mobile App, channel_USSD, etc., each 1 if that row belongs to that channel, 0 otherwise. This is necessary because models like Logistic Regression and Random Forest need numbers, not text categories.

drop_first=True — this drops one category per column (the first one alphabetically) rather than creating a column for every single value. Here's why that matters: if you have 6 channel columns and a transaction is 0 across all 6, that itself already tells you it must be the dropped 6th category — keeping all 6 would be redundant information (this redundancy is called the "dummy variable trap," and it specifically causes problems for Logistic Regression's math, though Random Forest tolerates it fine — dropping one is the safe default that works for both).

In [8]:
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

print("Shape before encoding:", df.shape)
print("Shape after encoding:", df_encoded.shape) #One-hot encoding the categoricals

Shape before encoding: (126000, 29)
Shape after encoding: (126000, 47)


shape jumped from 29 to 47 columns, meaning encoding added 18 new binary columns across account_type, kyc_tier, merchant_category, and channel combined (with one category dropped from each, per drop_first=True). That's in the right ballpark — merchant_category alone has 14 values, so ~13 new columns from that one field, plus a handful more from the other three.

Line 1 - for cat in categorical_features — loops through your four original category names: 'account_type', 'kyc_tier', 'merchant_category', 'channel'
col.startswith(cat + '_') — for a given column name, checks if it starts with that category name plus an underscore. E.g. does "channel_Mobile App" start with "channel_"? Yes. Does "amount_to_avg_ratio" start with "channel_"? No.
any(...) — checks if the column's name matches any of the four category prefixes (not just one specific one) — so it catches columns starting with account_type_, OR kyc_tier_, OR merchant_category_, OR channel_
for col in df_encoded.columns if ... — loops through every column in your newly encoded dataframe, keeping only the ones that passed the any(...) test above
The whole line, in plain English: "give me the names of every new column that pd.get_dummies created from our four categorical fields" — because after one-hot encoding, you don't know in advance exactly how many columns got created or what they're named (e.g. channel_Mobile App, channel_USSD, merchant_category_Travel, etc.) — this line finds them automatically rather than you having to type each one by hand.

Line 2 - This just combines two lists into one — your 7 original numeric column names, plus all the newly-discovered one-hot-encoded column names — into a single master list of every column that should go into the model.

Line 3 - This selects only those columns from df_encoded, dropping everything else (IDs, timestamps, status, is_fraud itself, etc.). X is now your feature matrix — literally the input the model will learn from.


Line 4 - Recall target = 'is_fraud' from earlier. This pulls out just that one column as its own separate object — y is your target/label, what the model is trying to predict. Note X and y are built as two separate objects on purpose: this is the standard shape scikit-learn (the library you'll use for LogReg/Random Forest) expects — features and labels always kept apart, never mixed in one table when training.

5 - 7 : Prints the dimensions of each (rows × columns for X, just a count for y, since it's a single column) and the full list of feature names — this is your final sanity check to see exactly what's about to be fed into the model, rather than trusting it blindly.

In [9]:
encoded_categorical_cols = [col for col in df_encoded.columns if any(col.startswith(cat + '_') for cat in categorical_features)]

feature_cols = numeric_features + encoded_categorical_cols

X = df_encoded[feature_cols]
y = df_encoded[target]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:", X.columns.tolist())

X shape: (126000, 29)
y shape: (126000,)

Feature columns: ['amount_to_avg_ratio', 'transaction_velocity_1h', 'is_new_device', 'has_device_on_record', 'is_cross_border', 'account_age_days', 'personal_spend_baseline_usd', 'account_type_Individual', 'kyc_tier_Tier2_Verified', 'kyc_tier_Tier3_Enhanced', 'merchant_category_Crypto Exchange', 'merchant_category_Electronics', 'merchant_category_Gambling/Gaming', 'merchant_category_Groceries', 'merchant_category_Healthcare', 'merchant_category_Insurance', 'merchant_category_P2P Transfer', 'merchant_category_Payroll Transfer', 'merchant_category_Restaurants', 'merchant_category_Retail', 'merchant_category_Subscription/SaaS', 'merchant_category_Travel', 'merchant_category_Utilities', 'merchant_category_Wire Transfer', 'channel_Card Not Present', 'channel_Card Present', 'channel_Mobile App', 'channel_USSD', 'channel_Web Dashboard']


In [10]:
model_ready = X.copy()
model_ready['is_fraud'] = y

model_ready.to_csv("../data/processed/finlora_model_ready.csv", index=False)
print("Saved:", model_ready.shape) #saving x and y combined asfinal model-ready dataset to CSV

Saved: (126000, 30)
